# Fine-tune the GroundCheck verifier on a free Colab GPU

This notebook builds the synthetic training data from the committed chunks, fine-tunes
**microsoft/deberta-v3-small** as a three-way verifier (supported / contradicted /
unsupported) with `scripts/train_verifier.py`, and downloads the model so
`FineTunedVerifier` can load it on your Mac. Nothing here calls a paid API.

**The hand-labeled gold set never enters training:** `build-training-data` excludes every
gold claim and every chunk retrieved for a candidate answer, and splits are by company
(`config/training.yaml`). For that exclusion to work, `data/eval/gold_labels.jsonl`,
`data/labels/labels.jsonl` and `data/candidates/` must be committed and pushed, together
with `data/chunks/*.jsonl`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Clone and install

For a private repo, add a `GITHUB_TOKEN` Colab secret (key icon on the left) with read access.

In [ ]:
import os
import subprocess

REPO = "krupal2241/GroundCheck"
BRANCH = "main"
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.isdir("GroundCheck"):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, "GroundCheck"], check=True
    )
%cd GroundCheck
!pip install -q -e ".[train]"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls data/chunks
!ls data/eval data/labels data/candidates 2>&1 | head -20

## 2. Settings

In [ ]:
BASE_MODEL = "microsoft/deberta-v3-small"
EPOCHS = 3
BATCH_SIZE = 16
SEED = 42
OUT = "models/verifier-deberta-v3-small"

## 3. Build the training data

Prints the examples per split and label, what was excluded, and how many examples each
rule produced. A WARNING about nothing to exclude means the gold files were not found:
stop and push them first (unless you have not labeled anything yet).

In [ ]:
!groundcheck build-training-data
import json
from pathlib import Path

manifest = json.loads(Path("data/training/manifest.json").read_text())
for key in [
    "counts_by_split_and_label",
    "counts_by_rule",
    "gold_claims_excluded",
    "gold_chunks_excluded",
    "candidate_chunks_excluded",
]:
    print(f"{key}: {manifest[key]}")
!head -c 1200 data/training/train.jsonl

## 4. Smoke test (about a minute): 64 examples, one epoch

In [ ]:
!python scripts/train_verifier.py --base-model {BASE_MODEL} --limit-train 64 --epochs 1 \
    --out /content/smoke --fp16

## 5. Train

Fixed seeds (`--seed`); the best epoch is picked on dev macro-F1. Writes the model,
`metrics.json` and a model card (`README.md`) to `OUT`.

In [ ]:
!python scripts/train_verifier.py --base-model {BASE_MODEL} --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --seed {SEED} --out {OUT} --fp16

In [ ]:
!cat {OUT}/metrics.json
!cat {OUT}/README.md

## 6. Evaluate on the gold set (optional)

Accuracy here is real (the gold labels are hand-confirmed); latency is the T4's, not
your Mac's, so re-run `eval-verifiers` on the Mac for the README's latency column.

In [ ]:
# Fails with a clear message if data/eval/gold_labels.jsonl is not in the repo.
!groundcheck eval-verifiers --verifier finetuned --finetuned-path {OUT} --device cuda \
    --results-dir results/colab
!head -20 results/colab/verifier_eval.md

## 7. Download the model

Unzip on your Mac into the repo (`models/` is gitignored, so it stays local) and run
`groundcheck eval-verifiers --verifier finetuned --finetuned-path models/verifier-deberta-v3-small`.

In [ ]:
!cd models && zip -q -r ../verifier.zip verifier-deberta-v3-small
from google.colab import files

files.download("verifier.zip")